
### **1. Data loading and text preprocessing**


1. Download the dataset from HuggingFace - PL-Guard

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
from datasets import load_dataset

token = '[REDACTED_HF_TOKEN]'

dataset = load_dataset("NASK-PIB/PL-Guard", 'test', token=token)
dataset_adv = load_dataset("NASK-PIB/PL-Guard", 'test_adversarial', token=token)

After downloading the dataset, we can explore its structure and contents.

In [ ]:
print(dataset.column_names)
print(dataset_adv.column_names)


### **2. Text representation - using embeddings**


Computer doesn't understand words. It understands numbers. Therefore, we need to convert the text data into numerical format using embeddings.
What is embedding? - Embedding is a way to represent words or phrases as vectors of numbers in a continuous vector space.

This allows us to capture semantic relationships between words based on their meanings and contexts.
To generate embeddings for our text data, we leverage the Sentence-BERT (SBERT) architecture. It is better suited for generating sentence-level embeddings compared to traditional BERT models, as SBERT was specifically designed so that the cosine distance between vectors reflects semantic similarity.

Specifically, we implement the `paraphrase-multilingual-MiniLM-L12-v2` model from the SentenceTransformers library—a smaller and faster version of SBERT that maintains high performance.

In [ ]:
from sentence_transformers import SentenceTransformer
import pandas as pd
import numpy as np

model_name = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'  #unique name of the model from HuggingFace
model = SentenceTransformer(
    model_name)  #download the model, create an instance of SentenceTransformer class (wraps complex functionality like tokenization, data flow into simple function), load the model


Tokenization is the process of breaking down text into smaller units called tokens (words, subwords, or characters, depending on the strategy). The tokens are then mapped to integer IDs using the model’s vocabulary. These IDs are converted into vectors via an embedding layer, and then passed through the Transformer blocks (self-attention, feed-forward/linear layers, and normalizations) to produce contextualized token embeddings. Next, a pooling operation (e.g., mean/max pooling, typically masking out padding tokens) aggregates token embeddings into a fixed-size sentence embedding, often followed by L2-normalization to keep embeddings on a consistent scale for similarity computations.

In [ ]:
print(model.get_sentence_embedding_dimension())
print(model)


In this model embeddings after pooling aren't normalized by default (norm isn't equal 1.0). Every sentence from this model will be converted into a vector of dimension 384. Normalization can be enabled via encode(normalize_embeddings=True)

In [ ]:
split = dataset['test']
sentences = split['text']
categories = split['category']

embeddings = model.encode(sentences, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=True)

In [ ]:
print(f'Shape of embeddings: {embeddings.shape}')
print(f'Example embedding vector: {embeddings[:5]}')

Embeddings shape is (900,384) that means we have 900 sentences and each sentence is represented by a vector of 384 dimensions. If two sentences are semantically similar, their embeddings will be close in the vector space.


### 3. Dimensionality reduction and visualization


Our embeddings are in 384-dimensional space. To visualize them, we need to reduce their dimensionality to 2D or 3D. We can use techniques like PCA (Principal Component Analysis), t-SNE (t-Distributed Stochastic Neighbor Embedding) or UMAP (Uniform Manifold Approximation and Projection) for this purpose. First, we will use PCA to reduce the embeddings to 2D.

#### 3.1 PCA dimensionality reduction

In [ ]:
from sklearn.decomposition import PCA

pca_2d = PCA(n_components=2)
embeddings_2d = pca_2d.fit_transform(embeddings)

pca_3d = PCA(n_components=3)
embeddings_3d = pca_3d.fit_transform(embeddings)

print(f'2D: {sum(pca_2d.explained_variance_ratio_) * 100:.2f}% variance explained by 2 components)')
print(f'3D: {sum(pca_3d.explained_variance_ratio_) * 100:.2f}% variance explained by 3 components)')

PCA is a linear projection and in our embeddings the first 2 components explain ~14% variance (3 components ~20%), which is typical for high-dimensional sentence embeddings. Therefore, 2D/3D PCA plots are useful mainly for visualization/sanity-check, and we will also try non-linear methods (UMAP/t-SNE) that can better preserve local neighborhood structure

In [ ]:
import plotly.express as px

fig = px.scatter(
    x=embeddings_2d[:, 0],  # first PCA component [take all rows, first column]
    y=embeddings_2d[:, 1],  # second PCA component
    color=categories,
    # hover_data=[sentences],
    title='PCA 2D of Sentence Embeddings',
    labels={'x': 'PCA Component 1', 'y': 'PCA Component 2'},
)

# 2. Poprawiamy czytelność - legenda i wielkość punktów
fig.update_traces(marker=dict(size=8, opacity=0.7))
fig.update_layout(legend_title_text='Kategorie złośliwości')

# 3. Wyświetlenie wykresu
fig.show()

#### 3.2 t-SNE dimensionality reduction

Now we'll check t-SNE for dimensionality reduction.

In [ ]:
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, perplexity=30, random_state=42,
            max_iter=1000)  #perplexity is related to the number of nearest neighbors that point takes into consideration
embeddings_tsne = tsne.fit_transform(embeddings)

fig_tsne = px.scatter(
    x=embeddings_tsne[:, 0],
    y=embeddings_tsne[:, 1],
    color=categories,
    title='t-SNE 2D of Sentence Embeddings',
    labels={'x': 't-SNE Component 1', 'y': 't-SNE Component 2'},
)

fig_tsne.show()

In [ ]:
embeddings_pca_50 = PCA(n_components=50, random_state=42).fit_transform(embeddings)
tsne_with_pca_50 = TSNE(n_components=2, perplexity=30, random_state=42, max_iter=1000)
embeddings_tsne_with_pca_50 = tsne_with_pca_50.fit_transform(embeddings_pca_50)

fig_tsne_with_pca_50 = px.scatter(
    x=embeddings_tsne_with_pca_50[:, 0],
    y=embeddings_tsne_with_pca_50[:, 1],
    color=categories,
    title='t-SNE 2D of Sentence Embeddings (PCA to 50D first)',
    labels={'x': 't-SNE Component 1', 'y': 't-SNE Component 2'},
)
fig_tsne_with_pca_50.show()

In [ ]:
embeddings_pca_150 = PCA(n_components=150, random_state=42).fit_transform(embeddings)
tsne_with_pca_150 = TSNE(n_components=2, perplexity=30, random_state=42, max_iter=1000)
embeddings_tsne_with_pca_150 = tsne_with_pca_150.fit_transform(embeddings_pca_150)

fig_tsne_150 = px.scatter(
    x=embeddings_tsne_with_pca_150[:, 0],
    y=embeddings_tsne_with_pca_150[:, 1],
    color=categories,
    title='t-SNE 2D of Sentence Embeddings (PCA to 150D first)',
    labels={'x': 't-SNE Component 1', 'y': 't-SNE Component 2'},
)
fig_tsne_150.show()

In [ ]:
from sklearn.manifold import trustworthiness


def T(orig, red, k):
    return trustworthiness(orig, red, n_neighbors=k)


rows = []
for k in [5, 10, 20]:
    rows.append({
        "k": k,

        # t-SNE directly (end-to-end 384 -> 2D)
        "tSNE_384→2D (end-to-end)": T(embeddings, embeddings_tsne, k),

        # PCA50 -> t-SNE (end-to-end and t-SNE-only)
        "PCA50→tSNE (end-to-end 384→2D)": T(embeddings, embeddings_tsne_with_pca_50, k),
        "PCA50→tSNE (tSNE-only 50→2D)": T(embeddings_pca_50, embeddings_tsne_with_pca_50, k),

        # PCA150 -> t-SNE (end-to-end and t-SNE-only)
        "PCA150→tSNE (end-to-end 384→2D)": T(embeddings, embeddings_tsne_with_pca_150, k),
        "PCA150→tSNE (tSNE-only 150→2D)": T(embeddings_pca_150, embeddings_tsne_with_pca_150, k),
    })

df_T = pd.DataFrame(rows)
display(df_T.round(4))


Trustworthiness measures how well local neighbourhoods (k-nearest neighbours) from the original space are preserved after dimensionality reduction. The score ranges from 0 to 1, where values closer to 1 mean better preservation of local structure.

Because we use pipelines such as PCA → t-SNE, we report trustworthiness in two ways. First, end-to-end (384D → 2D): we compare the final 2D map against the original 384D embedding space. This reflects the combined distortion introduced by all steps (e.g., PCA plus t-SNE). Second, t-SNE-only (input space → 2D): we compare the final 2D map against the actual input space of t-SNE (e.g., PCA-50 or PCA-150). This isolates the distortion introduced by t-SNE alone, given its input.

We evaluated trustworthiness for k = 5, 10, 20. For t-SNE directly on 384D embeddings the scores are 0.9532 / 0.9293 / 0.9045. For PCA-50 → t-SNE (end-to-end) the scores are 0.9646 / 0.9419 / 0.9189, which is the best end-to-end result for all tested k. For PCA-50 → t-SNE (t-SNE-only) the scores are 0.9737 / 0.9563 / 0.9359. For PCA-150 → t-SNE (end-to-end) the scores are 0.9559 / 0.9327 / 0.9070, and for PCA-150 → t-SNE (t-SNE-only) they are 0.9569 / 0.9340 / 0.9084.

Overall, pre-reducing embeddings with PCA to 50 dimensions improves the quality of the 2D t-SNE visualisation in terms of preserving local neighbourhood structure.


In [ ]:
X_cluster = embeddings_pca_50
X_plot = embeddings_tsne_with_pca_50

#### 3.3 UMAP dimensionality reduction

Now we'll check UMAP for dimensionality reduction.

In [ ]:
import umap

umap_model = umap.UMAP(n_components=2, n_neighbors=5, min_dist=0.1,
                       random_state=42)  #n_neighbors controls local vs global structure, min_dist controls spacing between points
embeddings_umap = umap_model.fit_transform(embeddings)

fig_umap = px.scatter(
    x=embeddings_umap[:, 0],
    y=embeddings_umap[:, 1],
    color=categories,
    title='UMAP 2D of Sentence Embeddings',
    labels={'x': 'UMAP Component 1', 'y': 'UMAP Component 2'},
)
fig_umap.show()

In [ ]:
def calculate_trustworthiness(original_embeddings, reduced_embeddings, n_neighbors):
    return trustworthiness(original_embeddings, reduced_embeddings, n_neighbors=n_neighbors)


for k in [5, 10, 20]:
    trust_umap = calculate_trustworthiness(embeddings, embeddings_umap, n_neighbors=k)
    print(f'Trustworthiness of UMAP 2D: {trust_umap:.4f}')

We evaluated neighborhood preservation using trustworthiness (k=5/10/20). For UMAP, the best configuration was n_neighbors=15 (0.9481/0.9326/0.9076). However, t-SNE preceded by PCA to 50D achieved higher trustworthiness (0.9646/0.9419/0.9189), indicating better preservation of local neighborhoods in our setting.

The best dimensionality reduction method for our sentence embeddings appears to be t-SNE applied after reducing dimensions to 50D with PCA. This combination achieves the highest trustworthiness scores across multiple neighborhood sizes (k=5,10,20), indicating it best preserves local structures compared to other methods tested (PCA alone, t-SNE directly on 384D, UMAP).


### **4. Clustering**

#### 4.1 K-Means Clustering

This algorithm partitions the data into K clusters by minimizing the variance within each cluster. It iteratively assigns points to the nearest cluster centroid and updates centroids based on the mean of assigned points.

But how we should choose the optimal number of clusters (K)? One common method is the "elbow method," which involves plotting the inertia (sum of squared distances from each point to its assigned cluster center) against different values of K. The idea is to look for an "elbow" point in the plot where the rate of decrease in inertia sharply changes, indicating that adding more clusters beyond this point yields diminishing returns in terms of reducing inertia.

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

inertia = []
inertia_full = []
k_range = range(1, 21)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_cluster)
    inertia.append(km.inertia_)

for k in k_range:
    km_full = KMeans(n_clusters=k, random_state=42, n_init=10)
    km_full.fit(embeddings)
    inertia_full.append(km_full.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(k_range, inertia, 'bx-')
plt.xlabel('Liczba klastrów (k)')
plt.ylabel('Inercja (Suma kwadratów odległości)')
plt.title('Metoda łokcia dla K-Means')
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(k_range, inertia_full, 'bx-')
plt.xlabel('Liczba klastrów (k)')
plt.ylabel('Inercja (Suma kwadratów odległości)')
plt.title('Metoda łokcia dla K-Means (pełne embeddingi 384D)')
plt.show()


In [ ]:
from sklearn.metrics import silhouette_score

scores = []
for k in range(2, 21):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_cluster)
    scores.append(silhouette_score(X_cluster, labels))

print(scores)

The silhouette score measures how well each sample fits within its assigned cluster compared to other clusters. For a point i, it compares (a) the average distance to points in its own cluster (cohesion) with (b) the average distance to points in the nearest neighboring cluster (separation). The score ranges from −1 to 1: values close to 1 indicate well-separated clusters, values near 0 indicate overlapping clusters, and negative values suggest that many points may be assigned to the wrong cluster.

The elbow method does not provide a clear, unambiguous choice of k in our case: the inertia decreases smoothly and the “flattening” point can be interpreted in different ways (e.g., only around k ≈ 3,6, 11,15). In addition, the silhouette scores obtained for K-Means are consistently low across the tested range of k (approximately 0.05–0.07), which suggests that the data do not form well-separated, compact clusters under the assumptions of K-Means. Therefore, instead of selecting k based on an uncertain elbow and weak silhouette values, we set k = 15 to match the number of dataset categories and to enable a direct comparison between the unsupervised clusters and the ground-truth labels.

In [ ]:
kmeans_15 = KMeans(n_clusters=15, random_state=42, n_init=10)
labels_kmeans_15 = kmeans_15.fit_predict(X_cluster)

In [ ]:
df_visualisation = pd.DataFrame({
    'tsne_x': embeddings_tsne_with_pca_50[:, 0],
    'tsne_y': embeddings_tsne_with_pca_50[:, 1],
    'category': categories,
    'kmeans_15': labels_kmeans_15.astype(str),
    "text": sentences
})

print(df_visualisation.shape)
print(df_visualisation.head())

In [ ]:
fig = px.scatter(
    df_visualisation,
    x='tsne_x',
    y='tsne_y',
    color='kmeans_15',
    symbol='category',  #true labels from PL-Guard dataset
    title='K-Means (k=15) clustering on PCA-50 embeddings, visualized with t-SNE (2D)',
    labels={'tsne_x': 't-SNE Component 1', 'tsne_y': 't-SNE Component 2'},
)
fig.show()

In [ ]:
fig = px.scatter(
    df_visualisation,
    x='tsne_x', y='tsne_y',
    color='category',
    hover_data=['kmeans_15'],
    title='t-SNE colored by true category (hover shows KMeans cluster)'
)
fig.show()


In [ ]:
fig = px.scatter(
    df_visualisation,
    x='tsne_x', y='tsne_y',
    color='kmeans_15',
    hover_data=['category'],
    title='t-SNE colored by KMeans cluster (hover shows true category)'
)
fig.show()


In [ ]:
df_visualisation['safety'] = df_visualisation['category'].apply(lambda c: 'safe' if c == 'safe' else 'unsafe')

fig = px.scatter(
    df_visualisation,
    x='tsne_x', y='tsne_y',
    color='kmeans_15',
    symbol='safety',
    hover_data=['category'],
    title='t-SNE: color=KMeans cluster, symbol=safe/unsafe'
)
fig.show()


In [ ]:
ct_km = pd.crosstab(df_visualisation['kmeans_15'], df_visualisation['category'])
purity_km = ct_km.max(axis=1).sum() / ct_km.values.sum()
print("purity (kmeans):", purity_km)
print(ct_km)


Using K-Means with k = 15, the clustering shows limited alignment with the ground-truth categories. The purity is ~0.553, meaning that on average only about 55% of samples in each cluster belong to the cluster’s dominant class.

This is consistent with the low silhouette scores observed earlier (≈ 0.05–0.07 across different k), suggesting that the embeddings do not form strongly separated, spherical clusters in the Euclidean sense assumed by K-Means.

Overall, K-Means provides only weakly separable clusters on this embedding space; clusters tend to mix multiple categories rather than cleanly recovering the dataset labels.

#### 4.2 Hierarchical Clustering

In [ ]:
from sklearn.cluster import AgglomerativeClustering

agglo = AgglomerativeClustering(n_clusters=15, linkage='ward')
labels_agglo = agglo.fit_predict(X_cluster)

df_visualisation['agglo_15'] = labels_agglo.astype(str)

In [ ]:
fig = px.scatter(
    df_visualisation,
    x='tsne_x',
    y='tsne_y',
    color='agglo_15',
    hover_data=['category'],
    title='t-SNE (2D) visualization colored by Agglomerative clusters (k=15, trained on PCA-50 embeddings)',
    labels={'tsne_x': 't-SNE Component 1', 'tsne_y': 't-SNE Component 2'},
)
fig.show()

In [ ]:
import pandas as pd

ct = pd.crosstab(df_visualisation['agglo_15'], df_visualisation['category'])
purity = ct.max(axis=1).sum() / ct.values.sum()
print("purity:", purity)
print(ct)


In [ ]:
from sklearn.metrics import silhouette_score

print(silhouette_score(X_cluster, labels_agglo))


Agglomerative clustering with Ward linkage and k = 15 yields very similar behavior. The purity is ~0.512, slightly lower than K-Means but only by a marginal amount.

The silhouette score is ~0.0686, which is also very low, indicating poor separation between clusters and substantial overlap in the embedding space.

In summary, Agglomerative (Ward) does not substantially improve clustering quality over K-Means for these embeddings; both methods produce clusters that only partially correspond to the labeled categories.

#### 4.3 DBSCAN Clustering

In [ ]:
from sklearn.cluster import DBSCAN

dbscan = DBSCAN(eps=0.5, min_samples=5,
                metric='euclidean')  #eps - how close points should be to be considered part of the same neighborhood, min_samples - minimum number of points to form a dense region
labels_dbscan = dbscan.fit_predict(X_cluster)

df_visualisation['dbscan'] = labels_dbscan.astype(str)

n_outliers = sum(labels_dbscan == -1)
print(f'Number of outliers detected by DBSCAN: {n_outliers}')

In [ ]:
df_visualisation['is_outlier'] = (df_visualisation['dbscan'] == '-1')

fig = px.scatter(
    df_visualisation,
    x='tsne_x', y='tsne_y',
    color='dbscan',  # klastry DBSCAN + '-1' jako outliery
    symbol='is_outlier',  # outliery innym symbolem
    hover_data=['category'],
    title='t-SNE (2D) visualization colored by DBSCAN labels (fit on PCA-50 embeddings; outliers highlighted)',
    labels={'tsne_x': 't-SNE Component 1', 'tsne_y': 't-SNE Component 2'}
)
fig.show()


In [ ]:
labels = labels_dbscan
n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
print("clusters:", n_clusters, "outliers:", n_outliers)


In [ ]:
for eps in [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8]:
    for ms in [3, 5, 7, 9]:
        labels = DBSCAN(eps=eps, min_samples=ms, metric='euclidean').fit_predict(X_cluster)
        n_out = (labels == -1).sum()
        n_cl = len(set(labels)) - (1 if -1 in labels else 0)
        print(f"eps={eps:>4}, min_samples={ms}: clusters={n_cl:>2}, outliers={n_out}")


DBSCAN did not produce a stable clustering structure for our embeddings: for small eps almost all points are labeled as noise, while for larger eps the majority of points collapse into a single cluster. Hence DBSCAN is not suitable for discovering multiple meaningful clusters in this setting.

In [ ]:
outliers = df_visualisation[df_visualisation['is_outlier']]
print("Outliers:", len(outliers))
outliers[['category']].value_counts().head(10)

outlier_rate = (df_visualisation['is_outlier']
                .groupby(df_visualisation['category'])
                .mean()
                .sort_values(ascending=False))
print(outlier_rate.head(10))


In [ ]:
# 1) wybierz outliery
outliers = df_visualisation[df_visualisation['is_outlier']].copy()

# 2) spróbuj wykryć kolumnę z tekstem
candidate_cols = ['text', 'sentence', 'content', 'original_text', 'prompt', 'query']
text_col = next((c for c in candidate_cols if c in outliers.columns), None)

if text_col is None:
    # fallback: znajdź pierwszą kolumnę typu object/string, która nie jest 'category'
    for c in outliers.columns:
        if c not in ['category', 'dbscan', 'is_outlier'] and outliers[c].dtype == 'object':
            text_col = c
            break

print("Detected text column:", text_col)

# 3) wypisz 10 losowych outlierów
display_cols = ['category', 'dbscan']
if text_col: display_cols.append(text_col)

print(outliers[display_cols].sample(min(2, len(outliers)), random_state=42).to_string(index=False))


DBSCAN identified 40 samples as noise/outliers. Most of them belong to the “safe” class, which suggests that these samples are more diverse and less densely clustered in the embedding space.

#### 4.4 HDBSCAN Clustering

In [ ]:
import hdbscan
import numpy as np

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=20,  # zacznij np. 15 (możesz dać 10/20)
    min_samples=2,  # jak w DBSCAN; większe = więcej outlierów
    metric='euclidean'  # przy znormalizowanych embeddingach OK
)

labels_hdb = clusterer.fit_predict(X_cluster)

n_outliers = np.sum(labels_hdb == -1)
n_clusters = len(set(labels_hdb)) - (1 if -1 in labels_hdb else 0)

print("HDBSCAN clusters:", n_clusters)
print("HDBSCAN outliers:", n_outliers)


In [ ]:
for mcs in [5, 10, 15, 20, 30]:
    for ms in [None, 1, 2, 5, 10]:
        clusterer = hdbscan.HDBSCAN(
            min_cluster_size=mcs,
            min_samples=ms,
            metric='euclidean'  # albo 'euclidean' jeśli embeddings są L2=1
        )
        labels = clusterer.fit_predict(X_cluster)
        n_out = np.sum(labels == -1)
        n_cl = len(set(labels)) - (1 if -1 in labels else 0)
        print(f"mcs={mcs:>2}, ms={str(ms):>4} -> clusters={n_cl:>2}, outliers={n_out}")

HDBSCAN (Hierarchical DBSCAN) is a density-based clustering method that can discover clusters of varying density and explicitly label points that do not belong to any dense region as noise (outliers, label = -1). Two key hyperparameters are: min_cluster_size, which sets the minimum size of a cluster, and min_samples, which controls how conservative the algorithm is when deciding whether a point is part of a dense region (larger values typically produce more outliers). In our grid search we observed the expected trade-off: settings that produce more clusters also tend to increase the number of outliers. The most “inclusive” configuration was min_cluster_size=5, min_samples=2, yielding 2 clusters and only 19 outliers, while a more detailed partition (min_cluster_size=5, min_samples=1) produced 5 clusters with 140 outliers.

In [ ]:
configs = [
    ("hdb_mcs5_ms2", 5, 2),
    ("hdb_mcs5_ms1", 5, 1),
    ("hdb_mcs10_ms1", 10, 1),
]

hover_cols = [c for c in ["category", "text"] if c in df_visualisation.columns]

for col_name, mcs, ms in configs:
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=mcs,
        min_samples=ms,
        metric="euclidean"
    )
    labels = clusterer.fit_predict(X_cluster)

    n_outliers = int(np.sum(labels == -1))
    n_clusters = int(len(set(labels)) - (1 if -1 in labels else 0))
    print(f"{col_name}: clusters={n_clusters}, outliers={n_outliers}")

    # dopisz do DF (tak jak robisz dla DBSCAN/KMeans)
    df_visualisation[col_name] = labels.astype(str)
    out_col = f"{col_name}_is_outlier"
    df_visualisation[out_col] = (labels == -1)

    # 3) Wykres 2D t-SNE: kolor=klaster, symbol=outlier
    fig = px.scatter(
        df_visualisation,
        x="tsne_x", y="tsne_y",
        color=col_name,
        symbol=out_col,
        hover_data=hover_cols,
        title=f"t-SNE (2D) visualization colored by HDBSCAN labels (fit on PCA-50), mcs={mcs}, ms={ms}",
        labels={"tsne_x": "t-SNE Component 1", "tsne_y": "t-SNE Component 2"}
    )
    fig.show()
